# randn-like-noise-source — ex2: audit a reparameterization for dtype leaks

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `randn-like-noise-source`. Running the final beacon cell reports progress against the `Generative: randn-like noise source` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: randn-like noise source` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`randn-like-noise-source`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "randn-like-noise-source"
DD_SUBTOPIC = "Generative: randn-like noise source"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `randn_like` noise source — deepening refresher

`t.randn_like(x)` produces standard-normal noise with the SAME shape, dtype, and device as `x`. The two-arg cousin `t.randn(*x.shape)` matches shape only — dtype defaults to `float32`, device defaults to CPU.

**The silent dtype leak.** If your encoder runs in `float64` (or mixed precision in `float16`), the `randn(*shape)` form returns `float32` noise. Then `mu + sigma * eps` either upcasts the whole pipeline to `float32` (silently losing precision) or — worse — raises a confusing type-promotion error deep in the loss.

**The audit pattern.** When inheriting a buggy reparameterization function, assert dtype propagation on the OUTPUT:
```python
z = reparam(mu, sigma)
assert z.dtype == sigma.dtype, f'reparam leaked dtype: {z.dtype} vs {sigma.dtype}'
```
If that assertion ever fires, the call site is using `randn(*shape)` instead of `randn_like`.

### Exercise 2 — audit a reparameterization for dtype leaks

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a reparameterization function by checking whether its output dtype matches the input sigma's dtype across float32 / float64 / float16, returning a structured leak report.
> Keywords: randn_like, dtype-audit, debug, reparameterization
> ```

**KCs targeted:** `randn-like-vs-randn-shape`, `audit-output-dtype`

Two reparam functions are provided in the stub:
- `reparam_buggy(mu, sigma)` uses `t.randn(*sigma.shape)` (hardcodes float32 / CPU).
- `reparam_fixed(mu, sigma)` uses `t.randn_like(sigma)` (inherits dtype + device).

Implement `ex2_dtype_audit(reparam_fn, dtypes)`. For each `dtype` in the input list:

1. Build `mu = t.zeros(8, 4, dtype=dtype)` and `sigma = t.ones(8, 4, dtype=dtype)`.
2. Call `z = reparam_fn(mu, sigma)`.
3. Record a triple `(dtype, z.dtype, leaked)` where `leaked` is `True` iff `z.dtype != dtype`.
4. Return a list of those triples — one per input dtype, in order.

The function MUST be agnostic to which reparam was passed; the test calls it with both `reparam_buggy` and `reparam_fixed` and expects the buggy one to leak on `float64` / `float16` while the fixed one never leaks.

Input: `reparam_fn` (callable), `dtypes` (list of `torch.dtype`).
Output: `list[tuple[torch.dtype, torch.dtype, bool]]`.

In [ ]:
def ex2_dtype_audit(reparam_fn, dtypes):
    report = []
    for dt in dtypes:
        mu = t.zeros(8, 4, dtype=dt)
        sigma = t.ones(8, 4, dtype=dt)
        z = reparam_fn(mu, sigma)
        leaked = (z.dtype != dt)
        report.append((dt, z.dtype, leaked))
    return report


<details><summary>Solution</summary>

```python
def ex2_dtype_audit(reparam_fn, dtypes):
    report = []
    for dt in dtypes:
        mu = t.zeros(8, 4, dtype=dt)
        sigma = t.ones(8, 4, dtype=dt)
        z = reparam_fn(mu, sigma)
        leaked = (z.dtype != dt)
        report.append((dt, z.dtype, leaked))
    return report
```

**Why float16 exposes the bug, not float32.** When sigma is `float32` and noise is `float32` (from `randn(*shape)`), everything is type-consistent — no leak. When sigma is `float16`, the multiplication `sigma * eps` promotes the result to `float32` (the wider of the two), and the output is silently upcast. Your downstream graph thinks it's training in `float16` but is actually running in `float32` for most of the loss path — kills mixed-precision speedup.

**Why float64 might or might not flag.** When sigma is `float64` and noise is `float32`, the multiplication promotes to `float64` — so the output `z.dtype` is `float64`, matching the input. The leak is INVISIBLE at the output dtype level (but the noise had less precision than the rest of the computation). This audit catches the visible-leak case (`float16` → `float32` upcast); the harder invisible case needs intermediate-dtype tracking, which is out of scope.

**The fix is mechanical.** Always use `randn_like(sigma)` for reparameterization noise — it inherits all three of (shape, dtype, device) and never leaks.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()